# 9 (Lanjutan) — Representasi Term dengan Word2Vec

**Tujuan praktikum**
1. Mahasiswa memahami representasi term terdistribusi (*word embedding*) dan perbedaannya dengan representasi *one-hot*/TF-IDF.
2. Mahasiswa mampu melatih Word2Vec (Skip-gram dan CBOW) dengan pustaka **gensim**.
3. Mahasiswa mampu memakai embedding untuk (a) query expansion, (b) *dense retrieval* dengan rata-rata vektor, dan (c) *hybrid retrieval* bersama BM25.
4. Mahasiswa mampu mengevaluasi dan membandingkan pendekatan tersebut dengan BM25 menggunakan MAP dan nDCG.

**Ide dasar (hipotesis distribusional):** kata yang muncul dalam konteks serupa cenderung bermakna serupa. Skip-gram melatih vektor $\vec v_w$ agar dapat memprediksi kata konteks $c$ di sekitar kata $w$:

$$P(c \mid w) = \frac{\exp(\vec u_c \cdot \vec v_w)}{\sum_{c'} \exp(\vec u_{c'} \cdot \vec v_w)}$$

Dalam praktik penyebutnya didekati dengan *negative sampling*. CBOW melakukan kebalikannya: memprediksi kata tengah dari rata-rata vektor konteksnya.

In [1]:
# Jalankan sekali jika pustaka belum terpasang:
# !pip install gensim pandas numpy matplotlib Sastrawi
import re, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from gensim.models import Word2Vec
from stki_utils import (corpus_texts, preprocess, BM25Model, TfidfModel,
                        load_queries, load_qrels, evaluate, load_corpus)

teks = corpus_texts()
docs = {d: preprocess(t) for d, t in teks.items()}
queries, qrels = load_queries(), load_qrels()
bm25 = BM25Model(docs)
run_bm25 = {q: [d for d, _ in bm25.search(preprocess(t), k=10)] for q, t in queries.items()}

ModuleNotFoundError: No module named 'gensim'

## Langkah 1 - Menyiapkan data latih
Word2Vec dilatih per kalimat. Kita memakai hasil preprocessing yang sama dengan indeks (stopword dihapus, stemming) agar term embedding cocok dengan term di indeks.

In [2]:
kalimat = [preprocess(s) for t in teks.values() for s in re.split(r"(?<=\.)\s+", t)]
kalimat = [k for k in kalimat if k]
print("Jumlah kalimat:", len(kalimat), "| jumlah token:", sum(map(len, kalimat)))
print(kalimat[:3])

Jumlah kalimat: 120 | jumlah token: 863
[['demam', 'darah', 'dengue'], ['demam', 'darah', 'dengue', 'sebab', 'virus', 'dengue', 'tular', 'lalu', 'gigit', 'nyamuk', 'aedes', 'aegypti'], ['gejala', 'utama', 'upa', 'demam', 'tinggi', 'dadak', 'nyeri', 'otot', 'bintik', 'merah', 'kulit']]


## Langkah 2 - Melatih Skip-gram dan CBOW
Parameter penting: `vector_size` (dimensi), `window` (lebar konteks), `sg` (1 = Skip-gram, 0 = CBOW), `min_count`, dan `epochs`. Karena korpus sangat kecil, kita memakai dimensi kecil dan epoch banyak. `workers=1`, `seed`, dan `hashfxn` tetap dipakai agar hasil dapat direproduksi.

In [3]:
def hash_stabil(s):
    return int(hashlib.md5(s.encode()).hexdigest()[:8], 16)

param = dict(vector_size=50, window=5, min_count=1, epochs=300, seed=42, workers=1, hashfxn=hash_stabil)
w2v_sg = Word2Vec(kalimat, sg=1, **param)
w2v_cbow = Word2Vec(kalimat, sg=0, **param)
print("Ukuran kosakata:", len(w2v_sg.wv), "| dimensi:", w2v_sg.wv.vector_size)
print("Vektor 'tani' (10 dimensi pertama):", w2v_sg.wv["tani"][:10].round(3))

Ukuran kosakata: 472 | dimensi: 50
Vektor 'tani' (10 dimensi pertama): [-0.02  -0.133 -0.532  0.121 -0.35  -0.199  0.234 -0.165 -0.391  0.253]


In [4]:
baris = []
for w in ["tani", "digital", "gizi", "harga", "aman", "ajar"]:
    baris.append({
        "term": w,
        "Skip-gram": ", ".join(f"{x} ({s:.2f})" for x, s in w2v_sg.wv.most_similar(w, topn=4)),
        "CBOW": ", ".join(f"{x} ({s:.2f})" for x, s in w2v_cbow.wv.most_similar(w, topn=4)),
    })
pd.DataFrame(baris).set_index("term")

,Skip-gram,CBOW
term,,
tani,"gabah (0.73), bulog (0.71), iot (0.71), serap ...","bulog (0.91), serap (0.90), gabah (0.90), stok..."
digital,"cakup (0.76), pustaka (0.76), literasi (0.76),...","pustaka (0.85), literasi (0.83), qris (0.83), ..."
gizi,"sekolah (0.93), status (0.92), tuju (0.92), kr...","sekolah (0.96), tuju (0.95), status (0.94), pe..."
harga,"naik (0.85), jual (0.85), tetap (0.84), pascap...","naik (0.94), sering (0.92), cabai (0.91), bera..."
aman,"berinternet (0.93), etika (0.92), cakup (0.91)...","berinternet (0.96), etika (0.94), cakup (0.94)..."
ajar,"misal (0.70), manajemen (0.70), kelas (0.68), ...","manajemen (0.83), konferensi (0.82), video (0...."


Tetangga terdekat mencerminkan kata yang muncul **bersama** di korpus (misalnya `gizi` dekat dengan `sekolah` dan `status`), bukan sinonim sejati. Dengan korpus hanya 120 kalimat, embedding lebih banyak menangkap co-occurrence lokal. Korpus besar (jutaan kalimat) diperlukan agar relasi semantik yang umum muncul.

## Langkah 3 - Visualisasi embedding
Vektor 50 dimensi diproyeksikan ke 2 dimensi dengan PCA (via SVD) dan diwarnai berdasarkan topik dokumen tempat term paling sering muncul.

In [5]:
def topik_dok(d):
    n = int(d[1:])
    return ["kesehatan", "pertanian", "teknologi", "ekonomi", "pendidikan"][(n - 1) // 6]

from collections import Counter
topik_term = {}
for d, tokens in docs.items():
    for t in tokens:
        topik_term.setdefault(t, Counter())[topik_dok(d)] += 1

terms = [t for t, _ in Counter(t for ts in docs.values() for t in ts).most_common(60)]
X = np.array([w2v_sg.wv[t] for t in terms])
X = X - X.mean(0)
U, S, Vt = np.linalg.svd(X, full_matrices=False)
P = X @ Vt[:2].T

warna = {"kesehatan": "tab:red", "pertanian": "tab:green", "teknologi": "tab:blue",
         "ekonomi": "tab:orange", "pendidikan": "tab:purple"}
plt.figure(figsize=(9, 6))
for (x, y), t in zip(P, terms):
    k = topik_term[t].most_common(1)[0][0]
    plt.scatter(x, y, color=warna[k], s=20)
    plt.annotate(t, (x, y), fontsize=8, xytext=(3, 2), textcoords="offset points")
for k, c in warna.items():
    plt.scatter([], [], color=c, label=k)
plt.legend(); plt.title("Proyeksi PCA embedding Skip-gram (60 term tersering)")
plt.grid(alpha=.3); plt.show()

## Langkah 4 - Query expansion dengan tetangga Word2Vec
Setiap term query ditambah `n` tetangga terdekat dengan bobot $w \cdot \text{similarity}$, lalu dicari dengan BM25 berbobot (Pertemuan 8).

In [6]:
wv = w2v_sg.wv

def w2v_expand(query, n=2, bobot=0.5):
    q_tokens = preprocess(query)
    hasil = {t: 1.0 for t in q_tokens}
    for t in q_tokens:
        if t in wv:
            for s, sim in wv.most_similar(t, topn=n):
                hasil.setdefault(s, bobot * sim)
    return hasil

print("Q2:", {t: round(w, 2) for t, w in w2v_expand(queries["Q2"]).items()})
run_qe = {q: [d for d, _ in bm25.search_weighted(w2v_expand(t))] for q, t in queries.items()}
evaluate(run_qe, qrels, k=5).loc[["Rata-rata"]]

Q2: {'teknologi': 1.0, 'tani': 1.0, 'modern': 1.0, 'efisiensi': 0.41, 'pencil': 0.4, 'gabah': 0.37, 'bulog': 0.36}


,P@5,R@5,F1@5,AP,nDCG@5
query_id,,,,,
Rata-rata,0.5,0.8125,0.6131,0.7696,0.8657


## Langkah 5 - Dense retrieval dengan rata-rata embedding
Dokumen dan query direpresentasikan sebagai **rata-rata vektor term**-nya, lalu diurutkan dengan cosine similarity. Varian kedua memberi bobot IDF pada setiap term agar kata umum tidak mendominasi.

In [7]:
idf = TfidfModel(docs).idf

def embed(tokens, pakai_idf=False):
    vs = [wv[t] * (idf.get(t, 1.0) if pakai_idf else 1.0) for t in tokens if t in wv]
    if not vs:
        return np.zeros(wv.vector_size)
    v = np.mean(vs, axis=0)
    return v / np.linalg.norm(v)

def dense_run(pakai_idf=False, k=10):
    D = {d: embed(t, pakai_idf) for d, t in docs.items()}
    run = {}
    for q, t in queries.items():
        qv = embed(preprocess(t), pakai_idf)
        skor = {d: float(v @ qv) for d, v in D.items()}
        run[q] = sorted(skor, key=lambda d: (-skor[d], d))[:k]
    return run

run_dense = dense_run(pakai_idf=False)
run_dense_idf = dense_run(pakai_idf=True)
print("Q3 (gizi anak balita):", run_dense["Q3"][:5], "| relevan:", qrels["Q3"])

Q3 (gizi anak balita): ['D02', 'D30', 'D04', 'D28', 'D03'] | relevan: {'D02': 2, 'D30': 2, 'D05': 1}


## Langkah 6 - Hybrid retrieval: BM25 + embedding
Skor BM25 dinormalisasi ke rentang 0–1 (dibagi skor maksimum) lalu digabung dengan cosine embedding:

$$\text{skor}(q,d) = \lambda \cdot \frac{\text{BM25}(q,d)}{\max_{d'} \text{BM25}(q,d')} + (1-\lambda) \cdot \cos(\vec q, \vec d)$$

In [8]:
D_idf = {d: embed(t, pakai_idf=True) for d, t in docs.items()}

def hybrid_run(lam, k=10):
    run = {}
    for q, t in queries.items():
        b = dict(bm25.search(preprocess(t), k=len(docs)))
        mx = max(b.values())
        qv = embed(preprocess(t), pakai_idf=True)
        skor = {d: lam * b.get(d, 0) / mx + (1 - lam) * float(D_idf[d] @ qv) for d in docs}
        run[q] = sorted(skor, key=lambda d: (-skor[d], d))[:k]
    return run

sweep = pd.DataFrame({lam: evaluate(hybrid_run(lam), qrels, k=5).loc["Rata-rata", ["AP", "nDCG@5"]]
                      for lam in [0.0, 0.3, 0.5, 0.7, 0.9]}).T
sweep.index.name = "lambda"
sweep.rename(columns={"AP": "MAP"})

,MAP,nDCG@5
lambda,,
0.0,0.7462,0.7788
0.3,0.8131,0.8793
0.5,0.8131,0.8793
0.7,0.8131,0.8793
0.9,0.8194,0.8811


$\lambda = 0$ berarti dense retrieval murni. Nilai $\lambda = 1$ tidak ditampilkan karena sama dengan BM25 (bandingkan dengan baris BM25 di Langkah 7).

## Langkah 7 - Perbandingan dengan BM25

In [9]:
ringkasan = pd.DataFrame({
    "BM25": evaluate(run_bm25, qrels, k=5).loc["Rata-rata"],
    "BM25 + QE Word2Vec": evaluate(run_qe, qrels, k=5).loc["Rata-rata"],
    "Dense Word2Vec (rata-rata)": evaluate(run_dense, qrels, k=5).loc["Rata-rata"],
    "Dense Word2Vec (rata-rata berbobot IDF)": evaluate(run_dense_idf, qrels, k=5).loc["Rata-rata"],
    "Hybrid BM25 + Word2Vec (lambda=0,7)": evaluate(hybrid_run(0.7), qrels, k=5).loc["Rata-rata"],
}).T.rename(columns={"AP": "MAP"})
ringkasan[["P@5", "MAP", "nDCG@5"]]

,P@5,MAP,nDCG@5
BM25,0.500,0.7756,0.8668
BM25 + QE Word2Vec,0.500,0.7696,0.8657
Dense Word2Vec (rata-rata),0.500,0.7482,0.7980
Dense Word2Vec (rata-rata berbobot IDF),0.500,0.7462,0.7788
"Hybrid BM25 + Word2Vec (lambda=0,7)",0.525,0.8131,0.8793


In [10]:
ax = ringkasan[["MAP", "nDCG@5"]].plot.barh(figsize=(8, 3.8))
ax.set_xlim(0, 1); ax.set_title("Word2Vec dibandingkan BM25"); ax.grid(axis="x", alpha=.3)
plt.tight_layout(); plt.show()

**Catatan hasil.** Dense retrieval murni dengan Word2Vec pada korpus sekecil ini belum mengalahkan BM25, karena embedding hanya dilatih dari 120 kalimat. Namun sebagai sinyal tambahan (hybrid), embedding membantu menemukan dokumen relevan yang tidak memuat kata query secara persis.

## Tugas 9
1. Bandingkan Skip-gram dan CBOW untuk dense retrieval dan hybrid. Mana yang lebih baik pada korpus ini?
2. Ubah `vector_size` (20, 50, 100) dan `window` (2, 5, 10). Laporkan pengaruhnya terhadap MAP dense retrieval.
3. Latih Word2Vec pada korpus yang jauh lebih besar (misalnya dump Wikipedia bahasa Indonesia atau korpus berita), lalu ulangi Langkah 4–7. Bandingkan tetangga terdekat `tani`, `digital`, dan `gizi` dengan hasil di atas.
4. Word2Vec tidak dapat membuat vektor untuk kata yang tidak ada di kosakata (OOV). Latih `gensim.models.FastText` dan tunjukkan bahwa kata yang salah ketik (misalnya `pertanain`) tetap memperoleh vektor.
5. Mengapa term yang di-stem justru dapat merugikan Word2Vec? Latih ulang tanpa stemming dan bandingkan.

In [11]:
# Tulis jawaban Anda di sini